# 🧬 NP-DNA Colab Worker
━━━━━━━━━━━━━━━━━━━━

Each Colab instance = 1 worker. Auto-assigns unique ID, trains on random data subset, uploads checkpoint.

**Flow:** ① Mount Drive ② Pull latest checkpoint ③ Train on T4 GPU ④ Upload result

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive mounted')

In [ ]:
!pip install -q torch numpy
import torch
import os, sys, json, random, shutil, time
print(f'🖥️  GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print(f'🎯 Device count: {torch.cuda.device_count()}')

In [ ]:
# ═══ CONFIG ═══
DRIVE_BASE = '/content/drive/MyDrive/npdna_training'
WORKERS_DIR = f'{DRIVE_BASE}/workers'
os.makedirs(WORKERS_DIR, exist_ok=True)

# Auto-assign unique worker ID
existing = [d for d in os.listdir(WORKERS_DIR) if d.startswith('worker_')]
used_ids = set()
for d in existing:
    try:
        used_ids.add(d.split('_')[1])
    except:
        pass
import string
worker_id = None
for letter in string.ascii_uppercase:
    if letter not in used_ids:
        worker_id = letter
        break
if not worker_id:
    worker_id = str(len(existing))

WORKER_DIR = f'{WORKERS_DIR}/worker_{worker_id}'
os.makedirs(WORKER_DIR, exist_ok=True)
with open(f'{WORKER_DIR}/STARTED', 'w') as f:
    f.write(f'Started at {time.ctime()}')
print(f'✅ Worker ID: {worker_id}')
print(f'   Drive: workers/worker_{worker_id}/')

In [ ]:
# ═══ LOAD CHECKPOINT ═══
CHECKPOINT_DIR = '/content/checkpoint'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
!cp -r "$DRIVE_BASE/base_checkpoint/"* "$CHECKPOINT_DIR/"
print(f'✅ Checkpoint loaded ({len(os.listdir(CHECKPOINT_DIR))} files):')
for f in os.listdir(CHECKPOINT_DIR):
    sz = os.path.getsize(f'{CHECKPOINT_DIR}/{f}') / 1e6
    print(f'   {f}: {sz:.1f}MB')

In [ ]:
# ═══ LOAD DATASET (random subset) ═══
DATA_DIR = '/content/data'
os.makedirs(DATA_DIR, exist_ok=True)

if os.path.exists(f'{DRIVE_BASE}/dataset'):
    for f in os.listdir(f'{DRIVE_BASE}/dataset'):
        src = os.path.join(DRIVE_BASE, 'dataset', f)
        if os.path.isfile(src):
            if f.endswith(('.jsonl', '.txt')):
                with open(src, 'r') as fin:
                    lines = fin.readlines()
                subset = random.sample(lines, max(1, len(lines) // 5))
                with open(f'{DATA_DIR}/{f}', 'w') as fout:
                    fout.writelines(subset)
                print(f' {f}: {len(subset)}/{len(lines)} lines sampled')
            else:
                shutil.copy2(src, f'{DATA_DIR}/{f}')
                sz = os.path.getsize(f'{DATA_DIR}/{f}') / 1e6
                print(f' {f}: {sz:.1f}MB copied')
    print(f'✅ Dataset ready at {DATA_DIR}')
else:
    print('⚠️  No dataset in Drive — training will use built-in data')

In [ ]:
# === INSTALL NP-DNA FROM DRIVE SOURCE ===
PROJECT_DIR = '/content/atulya_tantra'
SOURCE_DIR = f'{DRIVE_BASE}/source'
if os.path.exists(PROJECT_DIR):
    shutil.rmtree(PROJECT_DIR)
if os.path.exists(SOURCE_DIR):
    shutil.copytree(SOURCE_DIR, PROJECT_DIR)
    sys.path.insert(0, PROJECT_DIR)
    req = f'{PROJECT_DIR}/requirements.txt'
    if os.path.exists(req):
        !pip install -q -r {req}
    !pip install -q -e {PROJECT_DIR}
else:
    raise FileNotFoundError(f'Missing {SOURCE_DIR}. Run python colab_distributed/monitor.py --setup again.')
from tantra.npdna import NpDnaCore
from tantra.training import train
print('NP-DNA installed from Drive source')

In [ ]:
# ═══ TRAIN ═══
print('🚀 Starting training...')
device = 'cuda' if torch.cuda.is_available() else 'cpu'

CKPT_DIR = f'{WORKER_DIR}/checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)

def upload_callback(step, model):
    """Called periodically during training to upload checkpoint."""
    ckpt_path = f'{CKPT_DIR}/step_{step}.pt'
    torch.save(model.state_dict(), ckpt_path)
    print(f'  📤 Checkpoint step {step} → Drive ({os.path.getsize(ckpt_path)/1e6:.1f}MB)')

# Load model from checkpoint
core = NpDnaCore.from_checkpoint(CHECKPOINT_DIR)
core.to(device)

# ── TRAINING LOOP ──
# Adapt these parameters to your model & dataset
train(
    model=core,
    data_dir=DATA_DIR,
    steps=5000,                  # adjust for ~60 min Colab session
    batch_size=32,
    learning_rate=1e-4,
    checkpoint_callback=upload_callback,
    checkpoint_interval_mb=5000,  # upload every 5GB
    device=device,
)

print(f'✅ Training complete (worker {worker_id})')

In [ ]:
# ═══ UPLOAD FINAL CHECKPOINT ═══
final_path = f'{CKPT_DIR}/final.pt'
torch.save(core.state_dict(), final_path)
print(f'📤 Final checkpoint: {final_path}')
print(f'   Size: {os.path.getsize(final_path)/1e6:.1f}MB')

# Write DONE marker
with open(f'{WORKER_DIR}/DONE', 'w') as f:
    f.write(f'Completed at {time.ctime()}')

print()
print('═' * 40)
print(f'🏁 Worker {worker_id} FINISHED')
print('═' * 40)
print('\nThe local machine will auto-detect this on next sync.')
print('Your Telegram bot will notify you when merge happens.')

In [ ]:
# Optional: run another session with same worker
# Uncomment to loop immediately (use only if Colab session time allows)
# print('🔄 Starting next training cycle...')
# exec(open('/content/__main__.py').read())  # restart training from latest checkpoint